In [134]:
import pandas as pd
from unidecode import unidecode
import re
from collections import Counter
import os

In [135]:
# Set the directory where your CSV files are located
folder_path = 'data/all_wine_data_subsets'

# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Initialize an empty list to store individual dataframes
dfs = []

# Loop through each CSV file, read it into a dataframe, and append it to the list
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

# Concatenate all dataframes into one
wine_df = pd.concat(dfs, ignore_index=True)

## Inspect wine_df

In [136]:
wine_df.head()

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
0,https://www.danmurphys.com.au/product/DM_B9999...,Australian,Mixed Reds,This great value bundle is perfect for the sum...,Red,NaN,Multiple(Bundles),NaN,NaN,NaN,NaN,150.20,NaN,NaN,NaN
1,https://www.danmurphys.com.au/product/DM_99940...,Sherrah,Red et Al Grenache Shiraz Nero,Bush vine Grenache stars in this original blen...,Red,Other Red Blend,2020*,South Australia,McLaren Vale,Alex Sherrah,750mL,29.99,NaN,Medium Bodied,Dry
2,https://www.danmurphys.com.au/product/DM_99939...,Sherrah,Preservative Free Grenache,"Lifted and Luscious, bright fresh red berries ...",Red,Grenache,2021*,South Australia,McLaren Vale,Alex Sherrah,750mL,26.99,NaN,Medium Bodied,Dry
3,https://www.danmurphys.com.au/product/DM_99865...,Rabo de Gala,Rose,We welcome a brilliant rosé to our Portuguese ...,Red,Castelao and Cabernet Sauvignon.,NaN,NaN,NaN,NaN,750mL,11.99,NaN,NaN,NaN
4,https://www.danmurphys.com.au/product/DM_99831...,Vincent's Breeze,McLaren Vale Shiraz 2019,Vincent’s Breeze wines take their name from a ...,Red,Shiraz,2019,South Australia,McLaren Vale,NaN,750mL,24.99,NaN,Full Bodied,NaN


In [137]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6547 entries, 0 to 6546
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 6547 non-null   object 
 1   product_brand        6512 non-null   object 
 2   product_name         6512 non-null   object 
 3   product_description  6464 non-null   object 
 4   colour               6547 non-null   object 
 5   type                 6185 non-null   object 
 6   vintage              5634 non-null   object 
 7   state                4025 non-null   object 
 8   region               5215 non-null   object 
 9   winery               2407 non-null   object 
 10  size                 6309 non-null   object 
 11  non_member_price     6512 non-null   float64
 12  member_price         975 non-null    float64
 13  body                 4565 non-null   object 
 14  sweetness            4768 non-null   object 
dtypes: float64(2), object(13)
memory usage

In [138]:
wine_df.type.value_counts()

type
Shiraz                                            1046
Chardonnay                                         551
Cabernet Sauvignon                                 547
Pinot Noir                                         384
Sauvignon Blanc                                    292
                                                  ... 
White Varieties                                      1
Baga                                                 1
65% Pinot Noir 29% Chardonnay 6% Pinot Meunier       1
Grenache / Cinsault                                  1
Grenache, Tempranillo, Malbec                        1
Name: count, Length: 799, dtype: int64

## Clean type column

In [139]:
# drop null values
wine_df = wine_df.dropna(subset=["type"]).reset_index(drop=True)

In [140]:
# lowercase and remove accents
def clean_characters(type_str):
    if isinstance(type_str, str):  # Check if the value is a string
        return unidecode(type_str).lower()
    else:
        return type_str

# Apply the function to the 'type' column
wine_df['type'] = wine_df['type'].apply(clean_characters)

In [141]:
wine_df.type.value_counts()

type
shiraz                                            1052
chardonnay                                         555
cabernet sauvignon                                 548
pinot noir                                         386
sauvignon blanc                                    292
                                                  ... 
parellada macabeo xarel*lo                           1
white varieties                                      1
baga                                                 1
65% pinot noir 29% chardonnay 6% pinot meunier       1
grenache, tempranillo, malbec                        1
Name: count, Length: 758, dtype: int64

In [142]:
# remove non-words
def filter_words(type_str):
    if isinstance(type_str, str):
        # Remove any characters that are not letters or spaces
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', type_str)
        # Remove extra spaces and convert
        return ' '.join(cleaned_text.split())
    else:
        return type_str

wine_df['type'] = wine_df['type'].apply(filter_words)

In [143]:
# inspect the remaining type values
pd.set_option('display.max_rows', None)  # Set to None to display all rows
type_counts = wine_df['type'].value_counts()
type_counts_with_index = pd.DataFrame({'Wine Type': type_counts.index, 'Count': type_counts.values})
print(type_counts_with_index)
pd.reset_option('display.max_rows')

                                             Wine Type  Count
0                                               shiraz   1053
1                                           chardonnay    557
2                                   cabernet sauvignon    549
3                                           pinot noir    386
4                                      sauvignon blanc    292
5                                             riesling    214
6                                                 rose    185
7                                             grenache    111
8                                               merlot    110
9                                         pinot grigio    108
10                                         tempranillo    101
11                                          pinot gris     98
12                                            prosecco     88
13                               pinot noir chardonnay     85
14                                     cabernet merlot     84
15      

### rebuild type column

I will retrieve idividual type names from the data and use them to filter out ambiguous or sparse types. Finally I will rebuild the types to be more consistent in how they represent blends.

In [144]:
# manually retrieve individual wine types to create a type list
"""indices_to_retrieve = list(range(0, 8))  + list(range(9, 12)) + list(range(13, 16)) + [17] + list(range(22, 24)) \
    + list(range(26, 28)) + [29] + [31] + list(range(33, 37)) + list(range(39, 44)) + [49]
types_list = list(wine_df.type.value_counts().index)
filtered_types_list = [types_list[i] for i in indices_to_retrieve]
filtered_types_list = filtered_types_list + ["garnacha", "pinot meunier"]
filtered_types_list = filtered_types_list + ["mourvedre", "petit verdot", "brut", "mataro", "vranec", "gewurztraminer", "verdejo"] #  TODO validate these
filtered_types_list.sort()
#filtered_types_list = filtered_types_list + ["white", "red", "blend"] # add generic terms
filtered_types_list = ["sparkling"] + filtered_types_list # add generic terms
print(filtered_types_list)"""

'indices_to_retrieve = list(range(0, 8))  + list(range(9, 12)) + list(range(13, 16)) + [17] + list(range(22, 24))     + list(range(26, 28)) + [29] + [31] + list(range(33, 37)) + list(range(39, 44)) + [49]\ntypes_list = list(wine_df.type.value_counts().index)\nfiltered_types_list = [types_list[i] for i in indices_to_retrieve]\nfiltered_types_list = filtered_types_list + ["garnacha", "pinot meunier"]\nfiltered_types_list = filtered_types_list + ["mourvedre", "petit verdot", "brut", "mataro", "vranec", "gewurztraminer", "verdejo"] #  TODO validate these\nfiltered_types_list.sort()\n#filtered_types_list = filtered_types_list + ["white", "red", "blend"] # add generic terms\nfiltered_types_list = ["sparkling"] + filtered_types_list # add generic terms\nprint(filtered_types_list)'

In [145]:
# get filtered wine list from saved df
filtered_types_list = list(pd.read_csv("data/filtered_wine_types.csv").type.values)
filtered_types_list += ["montepulciano", "cinsault", "durif", "champagne", "marsanne", "barbera", "macabeo", "roussanne", "cuvee", "red blend", "white blend"]
print(filtered_types_list)

['sparkling', 'brut', 'cabernet sauvignon', 'chardonnay', 'chenin blanc', 'fiano', 'gamay', 'garnacha', 'gewurztraminer', 'glera', 'grenache', 'grillo', 'gruner veltliner', 'malbec', 'mataro', 'merlot', 'moscato', 'mourvedre', 'nebbiolo', 'nero davola', 'pet nat', 'petit verdot', 'pinot grigio', 'pinot gris', 'pinot meunier', 'pinot noir', 'prosecco', 'rice wine', 'riesling', 'rose', 'sangiovese', 'sauvignon blanc', 'semillon', 'shiraz', 'syrah', 'tempranillo', 'verdejo', 'verdelho', 'vermentino', 'viognier', 'vranec', 'montepulciano', 'cinsault', 'durif', 'champagne', 'marsanne', 'barbera', 'macabeo', 'roussanne', 'cuvee', 'red blend', 'white blend']


In [146]:
# save filtered wine types
#pd.DataFrame({"type":filtered_types_list}).to_csv('data/filtered_wine_types.csv', index=False)

#### Clean misspelling

In [147]:
wine_df['type'] = wine_df['type'].str.replace("piont", 'pinot', regex=True)
wine_df['type'] = wine_df['type'].str.replace("sauvingon", 'sauvignon', regex=True)
wine_df['type'] = wine_df['type'].str.replace("mouuvedre", 'mouvedre', regex=True)
wine_df['type'] = wine_df['type'].str.replace("sangiovesse", 'sangiovese', regex=True)
wine_df['type'] = wine_df['type'].str.replace("merot", 'merlot', regex=True)
wine_df['type'] = wine_df['type'].str.replace("melot", 'merlot', regex=True)
wine_df['type'] = wine_df['type'].str.replace("caberent", 'cabernet', regex=True)
wine_df['type'] = wine_df['type'].str.replace("semllion", 'semillon', regex=True)
wine_df['type'] = wine_df['type'].str.replace("chardonay", 'chardonnay', regex=True)

#### convert pinot grisgrigio

In [148]:
# convert pinot grigiogris to pinot gris or pinot grigio
wine_df[wine_df["type"] == "pinot grigiogris"]

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
1492,https://www.danmurphys.com.au/product/DM_22943...,South Island,White Mist Pinot Gris,The South Island of New Zealand is home to Mar...,White,pinot grigiogris,2022*,NaN,Marlborough,NaN,750mL,16.99,NaN,Light Bodied,Dry
1510,https://www.danmurphys.com.au/product/DM_21458...,Morialta Trails,Pinot Gris 750mL,We explored the Adelaide Hills to craft this w...,White,pinot grigiogris,2022,South Australia,Adelaide Hills,NaN,750mL,18.99,15.0,Light Bodied,Dry
1511,https://www.danmurphys.com.au/product/DM_21457...,Paradox,Pinot Gris 750mL,"Like the Paradox of life, an infinate mystery,...",White,pinot grigiogris,2022,NaN,South Eastern Australia,NaN,750mL,12.99,11.0,Light Bodied,Dry
1582,https://www.danmurphys.com.au/product/DM_16202...,Jean-Claude Mas,Le Pinot Grigio,Jean-Clause Mas has come up with a novel white...,White,pinot grigiogris,2021,NaN,Languedoc-Roussillon,NaN,750mL,19.99,16.0,Light Bodied,Dry
3096,https://www.danmurphys.com.au/product/DM_21480...,Sit Stay Society,Portia's 'Big Dog' Pinot Grigio,"Proudly Australian, Sit Stay Society celebrate...",White,pinot grigiogris,2022*,NaN,FLEURIEU Peninsula,NaN,750mL,15.99,13.0,Light Bodied,Dry


In [149]:
# replace pinot grisgrigio
wine_df.loc[wine_df["product_name"] == "White Mist Pinot Gris", "type"] = "pinot gris"
wine_df.loc[wine_df["product_name"] == "Pinot Gris 750mL", "type"] = "pinot gris"
wine_df.loc[wine_df["product_name"] == "Portia's 'Big Dog' Pinot Grigio", "type"] = "pinot grigio"

In [150]:
# convert pinot grigiogris to pinot gris or pinot grigio
wine_df[wine_df["type"] == "pinot grigiogris"]

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
1582,https://www.danmurphys.com.au/product/DM_16202...,Jean-Claude Mas,Le Pinot Grigio,Jean-Clause Mas has come up with a novel white...,White,pinot grigiogris,2021,NaN,Languedoc-Roussillon,NaN,750mL,19.99,16.0,Light Bodied,Dry


#### Clean cabernet type values

In [151]:
len(wine_df[wine_df["type"].str.contains("cabernet sauvignon")])

690

In [152]:
wine_df['type'] = wine_df['type'].str.replace("cabernet sauvignon", 'cabernet', regex=True)
wine_df['type'] = wine_df['type'].str.replace("cabernet", 'cabernet sauvignon', regex=True)

In [153]:
len(wine_df[wine_df["type"].str.contains("cabernet sauvignon")])

941

#### Clean pinot meunier type values

In [154]:
len(wine_df[wine_df["type"].str.contains("pinot meunier")])

56

In [155]:
wine_df['type'] = wine_df['type'].str.replace("pinot meunier", 'meunier', regex=True)
wine_df['type'] = wine_df['type'].str.replace("meunier", 'pinot meunier', regex=True)

In [156]:
len(wine_df[wine_df["type"].str.contains("pinot meunier")])

66

#### Clean muscat to muscato

In [157]:
len(wine_df[wine_df["type"].str.contains("moscato")])

84

In [158]:
wine_df['type'] = wine_df['type'].str.replace("muscat", 'moscato', regex=True)

In [159]:
len(wine_df[wine_df["type"].str.contains("moscato")])

96

#### Further expansion of words

In [160]:
print("n pinot gris:", len(wine_df[wine_df["type"].str.contains("pinot gris")]))
wine_df['type'] = wine_df['type'].str.replace("pinot gris", 'gris', regex=True)
wine_df['type'] = wine_df['type'].str.replace("gris", 'pinot gris', regex=True)
print("n pinot gris:", len(wine_df[wine_df["type"].str.contains("pinot gris")]))

print("n pinot noir:", len(wine_df[wine_df["type"].str.contains("pinot noir")]))
wine_df['type'] = wine_df['type'].str.replace("pinot noir", 'noir', regex=True)
wine_df['type'] = wine_df['type'].str.replace("noir", 'pinot noir', regex=True)
print("n pinot noir:", len(wine_df[wine_df["type"].str.contains("pinot noir")]))



n pinot gris: 112
n pinot gris: 116
n pinot noir: 608
n pinot noir: 617


#### start matching types

In [161]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6185 entries, 0 to 6184
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 6185 non-null   object 
 1   product_brand        6185 non-null   object 
 2   product_name         6185 non-null   object 
 3   product_description  6139 non-null   object 
 4   colour               6185 non-null   object 
 5   type                 6185 non-null   object 
 6   vintage              5450 non-null   object 
 7   state                3932 non-null   object 
 8   region               5095 non-null   object 
 9   winery               2373 non-null   object 
 10  size                 6117 non-null   object 
 11  non_member_price     6185 non-null   float64
 12  member_price         933 non-null    float64
 13  body                 4530 non-null   object 
 14  sweetness            4719 non-null   object 
dtypes: float64(2), object(13)
memory usage

In [162]:
wine_df.iloc[1227]

href                   https://www.danmurphys.com.au/product/DM_66765...
product_brand                                                    Tulloch
product_name                                                    Verdelho
product_description    Tulloch Verdelho is an exciting aromatic white...
colour                                                             White
type                                                            verdelho
vintage                                                              NaN
state                                                    New South Wales
region                                                     Hunter Valley
winery                                                       Jay Tulloch
size                                                               750mL
non_member_price                                                   17.99
member_price                                                        13.0
body                                               

In [163]:
"""all_matching_words = []
leftover_words = []
for i, row in wine_df.iterrows():
    matched = False
    # build matched types list
    row_matching_words = []
    for word in filtered_types_list:
        if word in row['type']:
            row_matching_words.append(word)
            matched = True

    # check for extra words
    if matched:
        row_type = row.copy()['type']
        for matched_word in row_matching_words:
            row_type = row_type.replace(matched_word, "_")
        remaining_words = re.findall(r'\b[a-zA-Z]+\b', row_type)
        if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words:
            row_matching_words.append("blend")
    else:
        row_type = row.copy()['type']
        remaining_words = []

    leftover_words.append(remaining_words)
    print(i, row_matching_words, row_type, remaining_words)
    print(i, row['type'])
    all_matching_words.append(row_matching_words)"""

'all_matching_words = []\nleftover_words = []\nfor i, row in wine_df.iterrows():\n    matched = False\n    # build matched types list\n    row_matching_words = []\n    for word in filtered_types_list:\n        if word in row[\'type\']:\n            row_matching_words.append(word)\n            matched = True\n\n    # check for extra words\n    if matched:\n        row_type = row.copy()[\'type\']\n        for matched_word in row_matching_words:\n            row_type = row_type.replace(matched_word, "_")\n        remaining_words = re.findall(r\'\x08[a-zA-Z]+\x08\', row_type)\n        if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words:\n            row_matching_words.append("blend")\n    else:\n        row_type = row.copy()[\'type\']\n        remaining_words = []\n\n    leftover_words.append(remaining_words)\n    print(i, row_matching_words, row_type, remaining_words)\n    print(i, row[\'type\'])\n    all_matching_words.append(row_matching_wo

In [164]:
all_matching_words = []
leftover_words = []
eval_list = [] # used for evaluation of words
for i, row in wine_df.iterrows():
    matched = False
    # build matched types list
    row_matching_words = []
    for word in filtered_types_list:
        if word in row['type']:
            row_matching_words.append(word)
            matched = True

    # check if blend
    row_type = row.copy()['type']
    for matched_word in row_matching_words:
        row_type = row_type.replace(matched_word, "_")
    remaining_words = re.findall(r'\b[a-zA-Z]+\b', row_type)
    #if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words and matched:
        #row_matching_words.append("blend")

    leftover_words.append(remaining_words)
    all_matching_words.append(row_matching_words)
    eval_list.append({"product_name": row["product_name"],
                      "full_string" : row['type'],
                      "extracted_words" : row_matching_words,
                      "leftover_string" : row_type,
                      "remaining_words" : remaining_words
                     })

#### investigate leftover type words

In [165]:
# investigate leftover words

# Flatten the list of lists
flat_list = [item for sublist in leftover_words for item in sublist]

# Count the occurrences of each string
word_counts = Counter(flat_list)

# Display the word counts
print(word_counts)

Counter({'other': 110, 'blend': 79, 'franc': 65, 'red': 58, 'blends': 52, 'white': 49, 'and': 47, 'wine': 44, 'gsm': 37, 'pinot': 34, 'sbs': 33, 'msg': 31, 'sgm': 31, 'ssb': 31, 'blanc': 30, 'de': 20, 'varietal': 16, 'field': 15, 'mixed': 12, 'blancs': 11, 'sauvignon': 9, 'primitivo': 8, 'dolcetto': 7, 'carignan': 7, 'albarino': 7, 'mencia': 7, 'saperavi': 7, 'rolle': 6, 'graciano': 6, 'varietals': 6, 'bianco': 6, 'sweet': 6, 'picpoul': 6, 'fino': 6, 'nero': 6, 'touriga': 6, 'assyrtiko': 5, 'di': 5, 'burgundy': 5, 'tinto': 5, 'ugni': 5, 'parellada': 5, 'carmenere': 5, 'monastrell': 5, 'zinfandel': 4, 'cabermet': 4, 'cab': 4, 'soave': 4, 'non': 4, 'pecorino': 4, 'cortese': 4, 'melon': 4, 'bourgogne': 4, 'arneis': 4, 'pink': 4, 'dry': 4, 'semillion': 4, 'arinto': 4, 'pinotage': 4, 'wines': 4, 'traminer': 4, 'chenin': 4, 'varieties': 4, 'of': 4, 'palomino': 4, 'sangria': 4, 'chianti': 4, 'tannat': 4, 'agiorgitiko': 3, 'rhone': 3, 'nerello': 3, 'mascalese': 3, 'clairette': 3, 'dolce': 3, '

In [166]:
# look into extracted and leftover words
for eval in eval_list:
    if "red" in eval["leftover_string"]:
        print("full_string:", eval["full_string"])
        print("leftover_string:", eval["leftover_string"])
        print("extracted_words:", eval["extracted_words"])
        print("remaining_words:", eval["remaining_words"])
        print()

full_string: red field blend
leftover_string: red field blend
extracted_words: []
remaining_words: ['red', 'field', 'blend']

full_string: rhone red
leftover_string: rhone red
extracted_words: []
remaining_words: ['rhone', 'red']

full_string: rhone red
leftover_string: rhone red
extracted_words: []
remaining_words: ['rhone', 'red']

full_string: rhone red
leftover_string: rhone red
extracted_words: []
remaining_words: ['rhone', 'red']

full_string: other red varietals
leftover_string: other red varietals
extracted_words: []
remaining_words: ['other', 'red', 'varietals']

full_string: other red varietals
leftover_string: other red varietals
extracted_words: []
remaining_words: ['other', 'red', 'varietals']

full_string: other red varietals
leftover_string: other red varietals
extracted_words: []
remaining_words: ['other', 'red', 'varietals']

full_string: other red varietal
leftover_string: other red varietal
extracted_words: []
remaining_words: ['other', 'red', 'varietal']

full_strin

In [95]:
wine_df["leftover_words"] = leftover_words

In [96]:
wine_df[wine_df["leftover_words"].apply(lambda x: len(x) > 0)]

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words
0,https://www.danmurphys.com.au/product/DM_99940...,Sherrah,Red et Al Grenache Shiraz Nero,Bush vine Grenache stars in this original blen...,Red,other red blend,2020*,South Australia,McLaren Vale,Alex Sherrah,750mL,29.99,NaN,Medium Bodied,Dry,"[other, red, blend]"
2,https://www.danmurphys.com.au/product/DM_99865...,Rabo de Gala,Rose,We welcome a brilliant rosé to our Portuguese ...,Red,castelao and cabernet sauvignon,NaN,NaN,NaN,NaN,750mL,11.99,NaN,NaN,NaN,"[castelao, and]"
4,https://www.danmurphys.com.au/product/DM_98985...,Thymiopoulos,'Young Vines' Xinomavro,Elegant character of fresh black fruit and her...,Red,xinomavro,Non Vintage,NaN,Macedonia,Apostolos Thymiopoulos,750ML,31.99,NaN,Full Bodied,NaN,[xinomavro]
5,https://www.danmurphys.com.au/product/DM_98984...,Gaia,Nemea Agiorgitiko 2017,"Rich and structured, it is balanced between fr...",Red,agiorgitiko,2019,NaN,Peloponnese,Yiannis Paraskevopoulous,750mL,38.99,NaN,Medium Bodied,NaN,[agiorgitiko]
21,https://www.danmurphys.com.au/product/DM_94320...,Piano dei Daini,Etna Rosso,The recovery of this old property started in 2...,Red,blend,2017*,NaN,Sicily,NaN,750mL,48.99,NaN,NaN,NaN,[blend]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,https://www.danmurphys.com.au/product/DM_56699...,Chateau L'Evangile,Pomerol 2012,It certainly helps when the parent company is ...,Red,red blend,2012,NaN,Bordeaux,NaN,750mL,405.99,NaN,NaN,Dry,"[red, blend]"
6170,https://www.danmurphys.com.au/product/DM_56697...,Château Ausone,St-Émilion 1er Grand Cru 'A' 2012,"Tasted blind at the 2012 Southwold tasting, th...",Red,bordeaux,2012,NaN,Bordeaux,NaN,750mL,1.00,1.0,NaN,Dry,[bordeaux]
6171,https://www.danmurphys.com.au/product/DM_56624...,Emilio Moro,Malleolus,Malleolus stands out from the very first momen...,Red,tinto fino,2018*,NaN,NaN,NaN,750mL,79.99,NaN,NaN,Dry,"[tinto, fino]"
6172,https://www.danmurphys.com.au/product/DM_56043...,Penfolds,Grange 2015,With an unbroken line of vintages from the exp...,Red,shiraz blend,2014,South Australia,Multi-Regional Blend,Peter Gago,750mL,799.00,NaN,Full Bodied,Dry,[blend]


#### add matching type words

In [97]:
# add type list as column
wine_df["type_as_list"] = all_matching_words

# re-write type column
wine_df['type'] = wine_df['type_as_list'].apply(lambda x: " ".join(x))

#remove rows with empty type value lists
wine_df = wine_df[wine_df["type_as_list"].apply(lambda x: len(x) > 0)].reset_index(drop=True)

In [98]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5586 entries, 0 to 5585
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 5586 non-null   object 
 1   product_brand        5586 non-null   object 
 2   product_name         5586 non-null   object 
 3   product_description  5551 non-null   object 
 4   colour               5586 non-null   object 
 5   type                 5586 non-null   object 
 6   vintage              4933 non-null   object 
 7   state                3715 non-null   object 
 8   region               4679 non-null   object 
 9   winery               2187 non-null   object 
 10  size                 5525 non-null   object 
 11  non_member_price     5586 non-null   float64
 12  member_price         849 non-null    float64
 13  body                 4146 non-null   object 
 14  sweetness            4305 non-null   object 
 15  leftover_words       5586 non-null   o

In [99]:
wine_df.head()

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words,type_as_list
0,https://www.danmurphys.com.au/product/DM_99939...,Sherrah,Preservative Free Grenache,"Lifted and Luscious, bright fresh red berries ...",Red,grenache,2021*,South Australia,McLaren Vale,Alex Sherrah,750mL,26.99,NaN,Medium Bodied,Dry,[],[grenache]
1,https://www.danmurphys.com.au/product/DM_99865...,Rabo de Gala,Rose,We welcome a brilliant rosé to our Portuguese ...,Red,cabernet sauvignon,NaN,NaN,NaN,NaN,750mL,11.99,NaN,NaN,NaN,"[castelao, and]",[cabernet sauvignon]
2,https://www.danmurphys.com.au/product/DM_99831...,Vincent's Breeze,McLaren Vale Shiraz 2019,Vincent’s Breeze wines take their name from a ...,Red,shiraz,2019,South Australia,McLaren Vale,NaN,750mL,24.99,NaN,Full Bodied,NaN,[],[shiraz]
3,https://www.danmurphys.com.au/product/DM_98492...,Morialta Trails,Adelaide Hills Pinot Rose,"This wine is named after the myriad of paths, ...",Red,rose,2020*,South Australia,Adelaide Hills,Morialta Trails,750mL,19.99,NaN,NaN,NaN,[],[rose]
4,https://www.danmurphys.com.au/product/DM_98107...,Minchinbury,Shiraz Cab 187mL,"This blend, sourced from premium Australian wi...",Red,shiraz,NaN,NaN,NaN,NaN,187mL,2.99,NaN,NaN,NaN,[],[shiraz]


In [100]:
wine_df.type.value_counts()

type
shiraz                                                   1066
cabernet sauvignon                                        620
chardonnay                                                568
pinot noir                                                412
sauvignon blanc                                           300
                                                         ... 
glera pinot noir                                            1
cabernet sauvignon chardonnay chenin blanc pinot noir       1
sparkling pinot noir                                        1
moscato sauvignon blanc                                     1
grenache malbec tempranillo                                 1
Name: count, Length: 189, dtype: int64

### Explore value counts of types

In [101]:
type_counts_df = pd.DataFrame(wine_df.type.value_counts())
type_counts_df.to_csv("data/wine_type_counts.csv")

In [102]:
print("number of rows where count of type is greater than 2:",type_counts_df[type_counts_df["count"] > 2]['count'].sum())

number of rows where count of type is greater than 2: 5460


In [103]:
type_counts_df[type_counts_df["count"] > 2]['count']

type
shiraz                                           1066
cabernet sauvignon                                620
chardonnay                                        568
pinot noir                                        412
sauvignon blanc                                   300
                                                 ... 
shiraz tempranillo                                  4
chardonnay semillon                                 4
cabernet sauvignon malbec merlot petit verdot       3
cabernet sauvignon malbec shiraz                    3
garnacha tempranillo                                3
Name: count, Length: 80, dtype: int64

In [104]:
print("number of distinct type values after filtering for greater than 2 occurences:",len(type_counts_df[type_counts_df["count"] > 2]['count']))

number of distinct type values after filtering for greater than 2 occurences: 80


### Save cleaned wine_df

In [105]:
wine_df

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words,type_as_list
0,https://www.danmurphys.com.au/product/DM_99939...,Sherrah,Preservative Free Grenache,"Lifted and Luscious, bright fresh red berries ...",Red,grenache,2021*,South Australia,McLaren Vale,Alex Sherrah,750mL,26.99,NaN,Medium Bodied,Dry,[],[grenache]
1,https://www.danmurphys.com.au/product/DM_99865...,Rabo de Gala,Rose,We welcome a brilliant rosé to our Portuguese ...,Red,cabernet sauvignon,NaN,NaN,NaN,NaN,750mL,11.99,NaN,NaN,NaN,"[castelao, and]",[cabernet sauvignon]
2,https://www.danmurphys.com.au/product/DM_99831...,Vincent's Breeze,McLaren Vale Shiraz 2019,Vincent’s Breeze wines take their name from a ...,Red,shiraz,2019,South Australia,McLaren Vale,NaN,750mL,24.99,NaN,Full Bodied,NaN,[],[shiraz]
3,https://www.danmurphys.com.au/product/DM_98492...,Morialta Trails,Adelaide Hills Pinot Rose,"This wine is named after the myriad of paths, ...",Red,rose,2020*,South Australia,Adelaide Hills,Morialta Trails,750mL,19.99,NaN,NaN,NaN,[],[rose]
4,https://www.danmurphys.com.au/product/DM_98107...,Minchinbury,Shiraz Cab 187mL,"This blend, sourced from premium Australian wi...",Red,shiraz,NaN,NaN,NaN,NaN,187mL,2.99,NaN,NaN,NaN,[],[shiraz]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5581,https://www.danmurphys.com.au/product/DM_54243...,Tim Adams,The Fergus Clare Valley Red Blend,The wine has lifted red berry aromas and flavo...,Red,grenache malbec tempranillo,2016*,South Australia,Clare Valley,Tim Adams,750mL,24.99,NaN,Full Bodied,Dry,[],"[grenache, malbec, tempranillo]"
5582,https://www.danmurphys.com.au/product/DM_54201...,Brown Brothers,Origins Series Malbec,This flavoursome wine is an ideal partner for ...,Red,malbec,2019*,Victoria,heathcote,Geoff Alexander,750mL,15.99,NaN,Medium Bodied,Dry,[],[malbec]
5583,https://www.danmurphys.com.au/product/DM_53699...,Penfolds,RWT Shiraz 2009,Don’t miss this rare chance to secure a magnif...,Red,shiraz,2009,NaN,Barossa Valley,NaN,750mL,279.00,NaN,NaN,NaN,[],[shiraz]
5584,https://www.danmurphys.com.au/product/DM_53699...,Penfolds,RWT Shiraz 2008,Don’t miss this rare chance to secure a magnif...,Red,shiraz,2008,South Australia,Barossa Valley,NaN,750mL,299.00,NaN,NaN,NaN,[],[shiraz]


In [106]:
wine_df.to_csv("data/cleaned_wine_df.csv", index=False)

In [107]:
for i, row in type_counts_df.iterrows():
    print(i, row["count"])

shiraz 1066
cabernet sauvignon 620
chardonnay 568
pinot noir 412
sauvignon blanc 300
riesling 220
rose 190
cabernet sauvignon merlot 145
grenache 122
chardonnay pinot noir 118
merlot 117
tempranillo 112
pinot grigio 109
pinot gris 103
cabernet sauvignon shiraz 95
prosecco rose 90
moscato 88
sauvignon blanc semillon 82
sangiovese 64
malbec 62
chardonnay pinot meunier pinot noir 59
nebbiolo 41
sparkling 40
semillon 36
grenache shiraz 32
vermentino 31
gamay 31
syrah 30
nero davola 25
montepulciano 25
verdelho 24
fiano 21
grenache syrah 18
pet nat 17
glera 17
cabernet sauvignon malbec 15
gruner veltliner 14
champagne 14
viognier 14
cabernet sauvignon merlot shiraz 14
garnacha 13
sparkling rose 13
rice wine 13
durif 12
chenin blanc 12
macabeo 11
brut 11
barbera 11
cabernet sauvignon merlot petit verdot 9
grenache mourvedre shiraz 9
grenache syrah cinsault 9
vranec 8
marsanne 8
shiraz viognier 7
grillo 6
verdejo 6
cabernet sauvignon sangiovese 6
gewurztraminer 5
malbec shiraz 5
mataro 5
cabe

In [108]:
print("unique type values:", len(set(wine_df.type)))

unique type values: 189
